In [ ]:
import openeo

import glob
import os

import rasterio
from rasterio.mask import mask
from rasterio.merge import merge

import geopandas as gpd
import numpy as np

from scipy.ndimage import median_filter

from collections import defaultdict

In [ ]:
# Remove connection to the backend and remove the refresh token 
from openeo.rest.auth.config import RefreshTokenStore
RefreshTokenStore().remove()

In [ ]:
year = 2016

## 1) Input setting and preparation

In [ ]:
start_date, end_date = ["2016-07-01", "2016-09-30"]

from pathlib import Path
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/" + str(year))
output_folder.mkdir(exist_ok=True)

In [ ]:
bursts_filename = '/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/S1_coh/s1_bursts_glacierSouthTyrol.geojson' # Extracted from the burst id map (https://sar-mpc.eu/sar-mpc-products/)
bursts = gpd.read_file(bursts_filename)
bursts.boundary.plot()

In [ ]:
url = "https://openeo.dataspace.copernicus.eu"
connection = openeo.connect(url).authenticate_oidc()

In [ ]:
#bursts = bursts.iloc[[5, 6,11,12, 14,15]]
#bursts  

## 2) Multi-temporal mosaic of coherence

For each burst compute the InSAR coherence time series and then the multitemporal max

In [ ]:
import time

job_list = []

for idx, b in bursts.iterrows():

    datacube = connection.datacube_from_process(
        "sentinel1_sar_coherence",
        namespace="https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/sentinel1_sar_coherence.json",
        temporal_extent=[start_date, end_date],
        temporal_baseline=12,
        burst_id=int(b["burst_id"]),
        sub_swath=b["subswath_name"],
        coherence_window_az=7,
        coherence_window_rg=22,
        polarization="VV",
    )

    datacube = (
        datacube
        .reduce_dimension(dimension="t", reducer="max")
        .save_result(format="GTiff")
    )

    job = datacube.create_job(
        title=f"sar_coherence_T{b['relative_orbit_number']}_{b['subswath_name']}_{b['burst_id']}_{year}"
    )

    job.start()
    job_list.append(job)

    print(f"Started {job.job_id}")

    time.sleep(10)   

## 3) Download the results

If in the same session where the jobs were launched, the results can be downloaded directly with:

In [ ]:
from datetime import datetime
job_create_date = datetime.strptime('2026-09-15T13:00:00Z', "%Y-%m-%dT%H:%M:%SZ")
job_list = [connection.job(j['id']) for j in connection.list_jobs() if datetime.strptime(j['created'], "%Y-%m-%dT%H:%M:%SZ") > job_create_date]
#job_list = job_list[1:]
print(len(job_list), "jobs created after", job_create_date)
job_list

In [ ]:
for i, j in enumerate(job_list):

    print(i, j.describe()["title"], j.status())

    if j.status() == "finished":

        title = j.describe()["title"]
        filepath = os.path.join(output_folder, f"{title}.tiff")

        print(f"Downloading: {filepath}")

        j.get_results().download_file(filepath)

        print("Done!")

## 4) Classify the glaciers

Reproject all to UTM32N with `gdalwarp`

In [ ]:
coh_max_filenames = []
for f in glob.glob(os.path.join(output_folder, 'sar_coherence_T*.tiff')):
    with rasterio.open(f) as src:
        target_filename = f[:-5] + '_EPSG32632.tiff'
        cmd = f'gdalwarp -t_srs EPSG:32632 -r bilinear -tr 10 10 {f} {target_filename}'
        os.system(cmd)
        coh_max_filenames.append(target_filename)
    print(f"{f}")

## 4.1) If the previous step was already done, load all files and composite files from one swadth

In [ ]:
# Find all coherence files for each swadth
files = list(output_folder.glob("sar_coherence_T*_*EPSG32632.tiff"))

# Extract the track number from the filename
groups = defaultdict(list)

for f in files:
    parts = f.stem.split("_")
    
    track = parts[2]  
    
    groups[track].append(f)

# Create one composite for each track
for track, filenames in groups.items():

    print(f"{track}: {len(filenames)} files")

    datasets = [rasterio.open(f) for f in filenames]

    mosaic, transform = merge(
        datasets,
        method="first",
        nodata=0
    )

    profile = datasets[0].profile.copy()
    profile.update(
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        count=1,
        nodata=0,
        dtype=mosaic.dtype
    )

    output_file = output_folder / f"sar_coherence_{track}_composite.tiff"

    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(mosaic[0], 1)

    for src in datasets:
        src.close()

    print(f"{output_file}")

### 5) Mask the outlines using the glacier inventory from 1997 and apply threshold

### 5.1) Create mosaic from descending and ascending swadth

In [ ]:
import re

files = list(output_folder.glob("sar_coherence_T*_composite.tiff"))

ascending_orbits = [15, 44, 117]
descending_orbits = [95, 168]

# Separate files
ascending_files = []
descending_files = []

for f in files:

    # Extract T number from filename
    match = re.search(r"T(\d+)", f.name)

    if match:
        orbit = int(match.group(1))

        if orbit in ascending_orbits:
            ascending_files.append(f)

        elif orbit in descending_orbits:
            descending_files.append(f)

print("Ascending:")
for f in ascending_files:
    print(f)

print("\nDescending:")
for f in descending_files:
    print(f)

In [ ]:
def merge_swaths(files, output_filename):

    datasets = [rasterio.open(f) for f in files]

    mosaic, transform = merge(
        datasets,
        method="max",
        nodata=0
    )

    meta = datasets[0].meta.copy()

    meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform,
        "count": 1,
        "nodata": 0
    })

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(mosaic)

    for ds in datasets:
        ds.close()

    print(f"Saved: {output_filename}")

In [ ]:
ascending_filename = output_folder / "coh_mosaic_ascending.tiff"
descending_filename = output_folder / "coh_mosaic_descending.tiff"

merge_swaths(
    ascending_files,
    ascending_filename
)

merge_swaths(
    descending_files,
    descending_filename
)

In [ ]:
ascending_filename = output_folder / "coh_mosaic_ascending.tiff"
descending_filename = output_folder / "coh_mosaic_descending.tiff"

mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"
glacier_inventory = gpd.read_file(mask_filename)


def threshold_and_clip(input_filename, output_filename):
    with rasterio.open(input_filename) as src:
        glacier_shapes = glacier_inventory.to_crs(src.crs).geometry.values

        clipped, clipped_transform = mask(
            src,
            shapes=glacier_shapes,
            crop=True,
            filled=False,
        )
        glacier = ((clipped[0] < 0.3) & ~clipped[0].mask).astype("uint8")

        output_meta = src.meta.copy()
        output_meta.update(
            {
                "height": glacier.shape[0],
                "width": glacier.shape[1],
                "transform": clipped_transform,
                "count": 1,
                "dtype": "uint8",
                "nodata": 0,
            }
        )

    with rasterio.open(output_filename, "w", **output_meta) as dst:
        dst.write(glacier, 1)

    print(f"Saved: {output_filename}")


threshold_and_clip(ascending_filename, output_folder / "masked_ascending_0.3.tif")
threshold_and_clip(descending_filename, output_folder / "masked_descending_0.3.tif")